In [1]:
!pip install -q -U langchain langchain-community langchain-text-splitters pypdf faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.9/132.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [10]:
!pip install -q sentence-transformers langchain-huggingface

In [14]:
!pip install -q transformers accelerate

In [6]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

loader = PyPDFDirectoryLoader("/content", glob="*.pdf")
documents = loader.load()

print(len(documents))

2011


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_spliter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap=200)
chunks = text_spliter.split_documents(documents)
print('total chunk :',len(chunks))

total chunk : 5769


In [11]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("FAISS Vector Store Created Successfully!")

FAISS Vector Store Created Successfully!


In [13]:
retriever = vectorstore.as_retriever(search_kwargs={'k':3})

In [21]:
from transformers import pipeline
llm = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", max_new_tokens= 200,  temperature=0.3)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [23]:
def ask_question(question):

    docs = retriever.invoke(question)

    context = "\n\n".join(doc.page_content for doc in docs)

    prompt = f"""
Use the context below to answer the question.
If the answer is not present in the context, say "I don't know."

Context:
{context}

Question:
{question}

Answer:
"""

    response = llm(prompt)[0]["generated_text"]

    answer = response.split("Answer:")[-1].strip()

    return answer

while True:
  question = input("\nAsk a question (type 'exit' to quit): ")
  if question.lower() == 'exit':
    print('seasion end')
    break
  answer = ask_question(question)

  print("\nAnswer:")
  print(answer)



Ask a question (type 'exit' to quit): What is supervised learning?


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
Supervised learning is a machine learning technique that involves training a model on a dataset
containing both features and labels. The model is then used to make predictions on new data.

Supervised learning is a subset of machine learning, which is a branch of artificial intelligence
that involves the use of algorithms to solve problems by analyzing and interpreting data.

Supervised learning is a popular approach in many fields, including computer vision, natural
language processing, and healthcare.

Supervised learning can be used to solve a wide range of machine learning problems, including:

1. Classification: predicting the class (e.g. “apple” or “banana”) of an input feature.

2. Regression: predicting a numeric output based on a set of input features.

3. Clustering: grouping similar data points together based on features.

4. Text analysis: identifying patterns and relationships in text data

Ask a question (type 'exit' to quit): What is a dictionary in Python?


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
The in operator works on dictionaries; it tells you whether something appears as a
key in the dictionary (appearing as a value is not good enough). The method values
returns the values as a type that can be converted to a list, and then you use the
in operator to check whether something appears as a value in a dictionary.

Ask a question (type 'exit' to quit): exit
seasion end
